# Introduction to cytoscape 

https://www.youtube.com/watch?v=g8xBlilTV4w

In [ ]:
import dash  # pip install dash
import dash_cytoscape as cyto  
import dash_html_components as html
from dash import dcc
from dash.dependencies import Output, Input
import pandas as pd  
import plotly.express as px


/var/folders/cl/bdrh3m8j0fdfj6zm7zhhc2ch0000gn/T/ipykernel_72087/2562119244.py:3: UserWarning: 
The dash_html_components package is deprecated. Please replace
`import dash_html_components as html` with `from dash import html`
  import dash_html_components as html
/var/folders/cl/bdrh3m8j0fdfj6zm7zhhc2ch0000gn/T/ipykernel_72087/2562119244.py:4: UserWarning: 
The dash_core_components package is deprecated. Please replace
`import dash_core_components as dcc` with `from dash import dcc`
  import dash_core_components as dcc


In [ ]:

external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css']
app = dash.Dash(__name__, external_stylesheets=external_stylesheets)

df = pd.read_csv("https://raw.githubusercontent.com/Coding-with-Adam/Dash-by-Plotly/master/Cytoscape/org-data.csv")

app.layout = html.Div([
    html.Div([
        cyto.Cytoscape(
            # will be used in the call back to make dashboard interactive
            id='org-chart',
            # Dictionary. Preset means you set position of nodes
            layout={'name': 'preset'},
            # Dictionary 
            style={'width': '100%', 'height': '500px'},
            # Body of the network, list of dictionaries with nodes and edges
            elements=[
                # Nodes elements
                # Keys: 
                # data - a dictionary itself with id - used in callbacks and label - what's going to be seen 
                # position - only if label is preset
                # locked - can't move anything
                # grabbable - can you grab it
                # selectable - can you select it
                # selected - automatically selected when app runs for first time
                # classes - a way to style app with different shapes 
                {'data': {'id': 'ed', 'label': 'Executive Director (Harriet)'},
                 'position': {'x': 150, 'y': 50},
                 'locked': True
                },

                {'data': {'id': 'vp1', 'label': 'Vice President (Sarah)'},
                 'position': {'x': 0, 'y': 150},
                 'grabbable': False
                },

                {'data': {'id': 'vp2', 'label': 'Vice President (Charlotte)'},
                 'position': {'x': 300, 'y': 150},
                'selectable': False
                },

                {'data': {'id': 'po1', 'label': 'Program Officer (Sojourner)'},
                 'position': {'x': -100, 'y': 250},
                 'selected': True
                },

                {'data': {'id': 'po2', 'label': 'Program Officer (Elizabeth)'},
                 'position': {'x': 150, 'y': 250}
                },

                {'data': {'id': 'pa', 'label': 'Program Associate (Ellen)'},
                 'position': {'x': 300, 'y': 350}
                },

                # Edge elements
                # data - source and target, where the edge starts and ends, label - label 
                {'data': {'source': 'ed', 'target': 'vp1', 'label': 'ED to VP1'}},
                {'data': {'source': 'ed', 'target': 'vp2'}},
                {'data': {'source': 'vp1', 'target': 'po1'}},
                {'data': {'source': 'vp1', 'target': 'po2'}},
                {'data': {'source': 'vp2', 'target': 'pa'}},
            ]
        )
    ], className='six columns'),

    html.Div([
        dcc.Graph(id='my-graph')
    ], className='six columns'),

], className='row')

# 
@app.callback(
    Output('my-graph','figure'),
    # tapNodeData - returns dictionary when you click on an element in the graph 
    Input('org-chart','tapNodeData'),
)
def update_nodes(data):
    if data is None:
        dff = df.copy()
        # make soujourner yellow initially because selected: True 
        dff.loc[dff.name == 'Program Officer (Sojourner)', 'color'] = "yellow"
        fig = px.bar(dff, x='name', y='slaves_freed')
        fig.update_traces(marker={'color': dff['color']})
        return fig
    else:
        print(data)
        dff = df.copy()
        # label set in the node, of the node selected 
        dff.loc[dff.name == data['label'], 'color'] = "yellow"
        print(dff)
        fig = px.bar(dff, x='name', y='slaves_freed')
        fig.update_traces(marker={'color': dff['color']})
        return fig


if __name__ == '__main__':
    app.run_server(debug=True)

    
# https://youtu.be/g8xBlilTV4w

# Dash Cytoscape Layout and user interaction 
https://www.youtube.com/watch?v=Ip2x7mmrBYY

## Layout

In [ ]:
import dash  # pip install dash
import dash_cytoscape as cyto  # pip install dash-cytoscape==0.2.0 or higher
import dash_html_components as html
import dash_core_components as dcc
from dash.dependencies import Output, Input
import pandas as pd  # pip install pandas
import plotly.express as px
import math

# Load extra layouts
# cyto.load_extra_layouts()

external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css']
app = dash.Dash(__name__, external_stylesheets=external_stylesheets)

df = pd.read_csv("https://raw.githubusercontent.com/Coding-with-Adam/Dash-by-Plotly/master/Cytoscape/org-data.csv")

# layouts: preset, random, cose, circular, grid, breadthfirst, concentric, external layouts
app.layout = html.Div([
    html.Div([
        dcc.Dropdown(
            id='dpdn',
            value='breadthfirst',
            clearable=False,
            options=[
                {'label': name.capitalize(), 'value': name}
                for name in ['breadthfirst' ,'grid', 'random', 'circle', 'cose', 'concentric']
            ]
        ),
        cyto.Cytoscape(
            # cytoscape reference chapter contains all options for these user interactions
            # used in call back 
            id='org-chart',
            # false = can grab nodes and move them around 
            autoungrabify=True,
            minZoom=0.2,
            maxZoom=1,
            #
            layout={'name': 'breadthfirst'},

            # layout={
            #     'name': 'circle',
            #     'radius': 250,
            #     'startAngle': math.pi * 1 / 9,
            #     'sweep': math.pi * 1 / 3
            # },

            # layout={
            #     'name': 'grid',
            #     'rows': 2,
            #     'cols': 4
            # },

            # breadth first tries to build out network like a tree with a root node 
            # layout={
            #     'name': 'breadthfirst', # tree
            #     'roots': '[id = "Executive Director (Harriet)"]'
            #     # 'roots': '#vp1, #vp2'
            #
            # },

            # external layouts are available but can slow dashboard down because many are algorithmically driven 

            style={'width': '100%', 'height': '500px'},
            elements=
                [
                    # Nodes elements
                    {'data': {'id': x, 'label': x}} for x in df.name
                ]
                +
                [
                    # Edge elements
                    {'data': {'source': 'Executive Director (Harriet)', 'target': 'Vice President (Sarah)'}},
                    {'data': {'source': 'Executive Director (Harriet)', 'target': 'Vice President (Charlotte)'}},
                    {'data': {'source': 'Vice President (Sarah)', 'target': 'Program Officer (Sojourner)'}},
                    {'data': {'source': 'Vice President (Sarah)', 'target': 'Program Officer (Elizabeth)'}},
                    {'data': {'source': 'Vice President (Charlotte)', 'target': 'Program Associate (Ellen)'}},
                ]
        )
    ], className='six columns'),

    html.Div([
        html.Div(id='empty-div', children='')
    ],className='one column'),

    html.Div([
        dcc.Graph(id='my-graph', figure=px.bar(df, x='name', y='slaves_freed'))
    ], className='five columns'),

], className='row')


@app.callback(Output('org-chart', 'layout'),
              Input('dpdn', 'value'))
def update_layout(layout_value):
    if layout_value == 'breadthfirst':
        return {
        'name': layout_value,
        'roots': '[id = "Executive Director (Harriet)"]',
        # animates changes 
        'animate': True
        }
    else:
        return {
            'name': layout_value,
            'animate': True
        }

## can use these types of things to track what users are doing with your graph 
@app.callback(
    Output('empty-div', 'children'),
    # whenever a mouse hovers over node you will see something 
    Input('org-chart', 'mouseoverNodeData'),
    Input('org-chart','mouseoverEdgeData'),
    Input('org-chart','tapEdgeData'),
    Input('org-chart','tapNodeData'),
    Input('org-chart','selectedNodeData')
)
def update_layout(mouse_on_node, mouse_on_edge, tap_edge, tap_node, snd):
    print("Mouse on Node: {}".format(mouse_on_node))
    print("Mouse on Edge: {}".format(mouse_on_edge))
    print("Tapped Edge: {}".format(tap_edge))
    print("Tapped Node: {}".format(tap_node))
    print("------------------------------------------------------------")
    print("All selected Nodes: {}".format(snd))
    print("------------------------------------------------------------")

    return 'see print statement for nodes and edges selected.'


@app.callback(
    Output('my-graph','figure'),
    Input('org-chart','tapNodeData'),
)
def update_nodes(data):
    if data is None:
        return dash.no_update
    else:
        dff = df.copy()
        dff.loc[dff.name == data['label'], 'color'] = "yellow"
        fig = px.bar(dff, x='name', y='slaves_freed')
        fig.update_traces(marker={'color': dff['color']})
        return fig


if __name__ == '__main__':
    app.run_server(debug=True)

    
# https://youtu.be/Ip2x7mmrBYY

# The Dash Callback - Input, Output, State, and more 
https://www.youtube.com/watch?v=mTsZL-VmRVE

In [ ]:
import pandas as pd
import plotly.express as px

import dash
import dash_core_components as dcc
import dash_html_components as html
from dash.dependencies import Input, Output, State

# Data source https://finance.yahoo.com  -Data owner: Stefano Leone on Kaggle
df = pd.read_csv("https://raw.githubusercontent.com/Coding-with-Adam/Dash-by-Plotly/master/Callbacks/Basic%20Callback/Mutual-Funds.csv")

colors = ["black", "blue", "red", "yellow", "pink", "orange"]

app = dash.Dash(__name__)

app.layout = html.Div(
    children=[
        # Most important part is the ids, they are used in the call back 
        dcc.Dropdown(id='my-dropdown', multi=True,
                     options=[{'label': x, 'value': x} for x in sorted(df.fund_extended_name.unique())],
                     value=["Fidelity 500 Index Fund", "Fidelity Advisor Freedom 2035 Fund Class A",
                            "Fidelity Freedom 2035 Fund"]),
        html.Button(id='my-button', n_clicks=0, children="Show breakdown"),
        # id = output component id, figuer = {} will take output component-property 'figure', must be a dictionary 
        dcc.Graph(id='graph-output', figure={}),

        html.Div(id="sentence-output", children=["This is the color I love"], style={}),
        dcc.RadioItems(id='my-radioitem', value="black", options=[{'label': c, 'value': c} for c in colors]),
    ]
)


# Single Input, single Output, State, prevent initial trigger of callback, PreventUpdate
@app.callback(
        # Every call back has output and input
    Output(component_id='graph-output', component_property='figure'),
    # whenever the component property of the input changes it triggers the callback so whenever the button 
    # is clicked we'll trigger the callback 
    #[Input(component_id='my-dropdown', component_property='value')],
     [Input(component_id='my-button', component_property='n_clicks')],
     # state does not trigger the callback, usually used when there are many options to change or with forms  
     [State(component_id='my-dropdown', component_property='value')],
     # prevents call back when page first loaded or page refreshed 
    prevent_initial_call=False
)
# after every callback you need a function, every function needs arguments that refer to the input and the state
# if you have one input you have one argument, two, two etc etc , arguments refer to component_property of input
# inital value chosen = ["Fidelity 500 Index Fund", "Fidelity Advisor Freedom 2035 Fund Class A","Fidelity Freedom 2035 Fund"])
def update_my_graph(n, val_chosen):
    if len(val_chosen) > 0:
        print(n)
        print(f"value user chose: {val_chosen}")
        print(type(val_chosen))
        # always make a copy, don't change the global df 
        dff = df[df["fund_extended_name"].isin(val_chosen)]
        fig = px.pie(dff, values="ytd_return", names="fund_extended_name", title="Year-to-Date Returns")
        fig.update_traces(textinfo="value+percent").update_layout(title_x=0.5)
        #whatever the function returns is equal to the component property of the output 
        return fig
    elif len(val_chosen) == 0:
        # if there is a bad value, don't update because don't return fig 
        raise dash.exceptions.PreventUpdate


# Multiple Input, multiple Output, dash.no_update
# @app.callback(
        # with multiple outputs, need to be inside list 
#     [Output('graph-output', 'figure'), Output('sentence-output', 'style')],
        # with multiple inputs need to be inside list 
#     [Input(component_id='my-radioitem', component_property='value'),
#      Input(component_id='my-dropdown', component_property='value')],
#     prevent_initial_call=False
# ) # with multiple inputs, multiple arguments 
# def update_graph(color_chosen, val_chosen):
#     if len(val_chosen) == 0:
        # do not update output if no values chosen , but do return new color 
        # need to return style as a dictionary because that's what the css expects 
        # dash.no_update allows you to not update one of many outputs while prevent update don't update output at all, usually used with one output
#         return dash.no_update, {"color": color_chosen}
#     else:
#         dff = df[df["fund_extended_name"].isin(val_chosen)]
#         fig = px.pie(dff, values="ytd_return", names="fund_extended_name", title="Year-to-Date Returns")
#         fig.update_traces(textinfo="value+percent").update_layout(title_x=0.5)
        # because there are two outputs, need to return two things because each return refers to the component-property of the out put  
        # need to return them in order 
#         return fig, {"color": color_chosen}


if __name__ == '__main__':
    app.run_server(debug=True)

    
# https://youtu.be/mTsZL-VmRVE

# Dash Cytoscape - Styling 
https://www.youtube.com/watch?v=iuHFwHgQIwg

In [ ]:
import dash  # pip install dash
import dash_cytoscape as cyto  # pip install dash-cytoscape==0.2.0 or higher
import dash_html_components as html
import dash_core_components as dcc
from dash.dependencies import Output, Input
import pandas as pd  # pip install pandas

df = pd.read_csv("https://raw.githubusercontent.com/Coding-with-Adam/Dash-by-Plotly/master/Cytoscape/org-data.csv")

external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css']
app = dash.Dash(__name__, external_stylesheets=external_stylesheets)

my_elements=[
    # Nodes elements
    {'data': {'id': 'ed', 'label': 'Executive Director (Harriet)'},
     'classes': 'purple' # One class
     },

    {'data': {'id': 'vp1', 'label': 'Vice President (Sarah)'},
     # used as reference for style 
     'classes': 'square' # One class
     },

    {'data': {'id': 'vp2', 'label': 'Vice President (Charlotte)'},
     'classes': 'square' # One class
     },

    {'data': {'id': 'po1', 'label': 'Program Officer (Sojourner)'},
     'classes': 'green diamond'  # Multiple classes
     },

    {'data': {'id': 'po2', 'label': 'Program Officer (Elizabeth)'},
     'classes': 'green diamond ' # Multiple classes
     },

    {'data': {'id': 'pa', 'label': 'Program Associate (Ellen)'},
     'classes': 'myimage' # One class
     },

    # Edge elements
    {'data': {'source': 'ed', 'target': 'vp1', 'weight': 1}, 'classes': 'purple'},
    {'data': {'source': 'ed', 'target': 'vp2', 'weight': 2}},
    {'data': {'source': 'vp1', 'target': 'po1', 'weight': 3}},
    {'data': {'source': 'vp1', 'target': 'po2', 'weight': 4}},
    {'data': {'source': 'vp2', 'target': 'pa', 'weight': 5}}
]

app.layout = html.Div([
    html.Div([
        dcc.Dropdown(
            id='dpdn',
            value='breadthfirst',
            clearable=False,
            options=[
                {'label': name.capitalize(), 'value': name}
                for name in ['breadthfirst' ,'grid', 'random', 'circle', 'cose', 'concentric']
            ]
        ),
        cyto.Cytoscape(
            id='org-chart',
            minZoom=0.2,
            maxZoom=2,
            layout={'name': 'breadthfirst'},
            elements=my_elements,
            style={'width': '100%', 'height': '500px'},
            # list of dictionaries, Group selectors or  class selectors 
            stylesheet=[
                # Group selectors for NODES
                    # used to annotate edges and nodes as a whole 
                    # two possible keys, selector and style 
                    # selector can be node or edge 
                    # style is a new dictionary the key is label and value is label to be displayed 
                    # https://js.cytoscape.org/#style/node-body
                    # https://js.cytoscape/#style/edge-line 
                {
                    'selector': 'node',
                    'style': {
                        'label': 'data(label)' # data(committee_name), data(committee_type) etc 
                    }
                },

                # Group selectors for EDGES
                {
                    'selector': 'edge',
                    'style': {
                        'label': 'data(weight)'
                    }
                },

                # Class selectors
                # to style network 
                # two possible keys, selector and style 
                # selector can be any class that was assigned to the elements' classes 
                {
                    'selector': '.purple',
                    'style': {
                        'background-color': 'purple',
                        'line-color': 'purple'
                    }
                },
                {
                    'selector': '.square',
                    'style': {
                        'shape': 'square',
                    }
                },
                {
                    'selector': '.myimage',
                    'style': {
                        'width': 100,
                        'height': 100,
                        'background-image': ['./assets/sunny-and-cloud.jpg']
                    }
                },
                {
                    'selector': '.green',
                    'style': {
                        'background-color': 'green',
                        'line-color': 'green'
                    }
                },
                {
                    'selector': '.diamond',
                    'style': {
                        'shape': 'diamond',
                    }
                },

                # Conditional Styling
                # this weight class only exists within the EDGES
                {
                    'selector': '[weight > 3]',
                    'style': {
                        'width': 20
                    }
                },
                # *= means string contains...
                {
                    'selector': '[label *= "rah"]',
                    'style': {
                        'background-color': '#000000',
                    }
                }
            ]
        )
    ], className='six columns'),

], className='row')


@app.callback(Output('org-chart', 'layout'),
              Input('dpdn', 'value'))
def update_layout(layout_value):
    if layout_value == 'breadthfirst':
        return {
        'name': layout_value,
        'roots': '[id = "ed"]',
        'animate': True
        }
    else:
        return {
            'name': layout_value,
            'animate': True
        }



if __name__ == '__main__':
    app.run_server(debug=True, port=4000)

    
# https://youtu.be/iuHFwHgQIwg
    